# Ant Colony Optimization (ACO) for Traveling Salesman Problem (TSP)

This Colab notebook demonstrates the Ant Colony Optimization (ACO) algorithm applied to a Traveling Salesman Problem (TSP). It includes functionalities for defining a graph, running the ACO algorithm, and visualizing the results, including pheromone trails and the best-found tour.

A key feature showcased in the second part of the script is the ACO algorithm's ability to efficiently re-compute an optimal solution when an edge in the graph is removed, leveraging the existing pheromone matrix.




**Key Concepts of ACO**:
- Pheromone Trails: Ants deposit a chemical substance called pheromone on the ground, marking paths. Shorter paths accumulate more pheromone, making them more attractive to subsequent ants.

- Probabilistic Tour Construction: Ants probabilistically choose the next city to visit based on the amount of pheromone on the edges and the desirability (inverse of distance) of the edges.

- Pheromone Evaporation: Pheromone trails gradually evaporate over time, preventing premature convergence to suboptimal solutions and allowing the exploration of new paths.

- Pheromone Deposition: After completing a tour, ants deposit pheromone inversely proportional to the length of their tour. Shorter tours deposit more pheromone.


Problem Setup We will define a graph representing cities (nodes) and the distances between them (edges).


In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math
from IPython.display import display, clear_output

Node Class Definition The Node class represents a single city or point in our Traveling Salesman Problem. Each node is defined by its two-dimensional coordinates (x and y).

In [2]:
class Node:
    """Represents a city/node in the graph."""
    def __init__(self, x, y):
        self.x = x
        self.y = y

Graph Class Definition The Graph class encapsulates the entire problem domain. It holds all the Node objects and an adjacency matrix (edges) representing the distances (weights) between them. An edge weight of np.inf indicates that there is no direct connection between two nodes.

In [3]:
class Graph:
    """Represents the entire graph with nodes and edges."""
    def __init__(self, x_coords, y_coords):
        self.n = len(x_coords)  # Number of nodes
        self.nodes = [Node(x_coords[i], y_coords[i]) for i in range(self.n)]
        self.edges = np.full((self.n, self.n), np.inf)  # Adjacency matrix for edge weights

    def add_edge(self, node1_idx, node2_idx, weight):
        """Adds an edge between two nodes with a given weight."""
        # Adjust indices to be 0-based for Python
        self.edges[node1_idx - 1, node2_idx - 1] = weight
        self.edges[node2_idx - 1, node1_idx - 1] = weight # Assuming an undirected graph



Graph Creation Function The create_graph function initializes the problem instance by defining the coordinates of all cities and setting up the predefined edge weights between them. These weights represent the "cost" or "distance" of traveling between two connected cities.

In [4]:
def create_graph(x_coords, y_coords):
    """
    Initializes the graph with given coordinates and predefined edge weights.
    """
    graph = Graph(x_coords, y_coords)

    # Add edges with predefined weights (as per MATLAB script)
    graph.add_edge(1, 2, 8)
    graph.add_edge(1, 3, 3)
    graph.add_edge(1, 4, 5)
    graph.add_edge(2, 3, 6)
    graph.add_edge(2, 5, 1)
    graph.add_edge(2, 6, 6)
    graph.add_edge(3, 4, 4)
    graph.add_edge(3, 6, 6)
    graph.add_edge(3, 7, 5)
    graph.add_edge(4, 7, 4)
    graph.add_edge(4, 8, 5)
    graph.add_edge(5, 6, 1)
    graph.add_edge(5, 9, 9)
    graph.add_edge(5, 10, 4)
    graph.add_edge(6, 7, 7)
    graph.add_edge(6, 10, 2)
    graph.add_edge(6, 11, 9)
    graph.add_edge(7, 8, 1)
    graph.add_edge(7, 11, 10)
    graph.add_edge(7, 12, 4)
    graph.add_edge(8, 12, 5)
    graph.add_edge(8, 13, 10)
    graph.add_edge(9, 10, 8)
    graph.add_edge(9, 14, 7)
    graph.add_edge(10, 11, 5)
    graph.add_edge(10, 14, 7)
    graph.add_edge(10, 15, 2)
    graph.add_edge(11, 12, 9)
    graph.add_edge(11, 15, 8)
    graph.add_edge(11, 16, 7)
    graph.add_edge(12, 13, 3)
    graph.add_edge(12, 16, 8)
    graph.add_edge(12, 17, 5)
    graph.add_edge(13, 17, 9)
    graph.add_edge(14, 15, 6)
    graph.add_edge(14, 18, 2)
    graph.add_edge(15, 16, 8)
    graph.add_edge(15, 18, 7)
    graph.add_edge(15, 19, 9)
    graph.add_edge(16, 17, 2)
    graph.add_edge(16, 19, 9)
    graph.add_edge(16, 20, 3)
    graph.add_edge(17, 20, 5)
    graph.add_edge(18, 19, 10)
    graph.add_edge(18, 21, 4)
    graph.add_edge(19, 20, 9)
    graph.add_edge(19, 21, 3)
    graph.add_edge(20, 21, 8)

    return graph


Fitness Function The fitness_function evaluates the quality of a given tour. For the TSP, fitness is typically defined as the total length of the tour. A lower fitness value indicates a better (shorter) tour.

In [5]:
def fitness_function(tour, graph):
    """
    Calculates the total length (fitness) of a given tour.
    A tour is a sequence of node indices (0-based).
    """
    fitness = 0
    for i in range(len(tour) - 1):
        current_node = tour[i]
        next_node = tour[i+1]
        fitness += graph.edges[current_node, next_node]
    return fitness

Roulette Wheel Selection The roulette_wheel function implements a probabilistic selection mechanism. In ACO, this is used by ants to choose the next city to visit based on the calculated probabilities (influenced by pheromone and desirability). Paths with higher probabilities have a larger "slice" of the roulette wheel and are more likely to be selected.

In [6]:
def roulette_wheel(probabilities):
    """
    Selects an item based on its probability using the roulette wheel selection method.
    """
    cumsum_probabilities = np.cumsum(probabilities)
    r = np.random.rand()
    # Find the first index where r is less than or equal to the cumulative sum
    next_node_idx = np.where(r <= cumsum_probabilities)[0][0]
    return next_node_idx

Ant Class Definition The Ant class represents an individual agent (ant) in the colony. Each ant maintains its own tour (a sequence of visited nodes) and its calculated fitness (total tour length).

In [7]:
class Ant:
    """Represents an individual ant and its tour."""
    def __init__(self):
        self.tour = []
        self.fitness = np.inf

Colony Class Definition The Colony class manages a collection of Ant objects. It also keeps track of the queen ant, which represents the best tour found by any ant in the colony so far.

In [8]:
class Colony:
    """Manages a collection of ants."""
    def __init__(self, ant_no):
        self.ants = [Ant() for _ in range(ant_no)]
        self.queen = Ant() # The best ant found so far

Create Colony Function The create_colony function is responsible for guiding each ant in the colony to construct a complete tour. Ants probabilistically choose their next city based on the pheromone levels on the edges and the desirability of those edges (inverse of distance). This process ensures that ants explore different paths while favoring those with higher pheromone concentrations.

In [9]:
def create_colony(graph, ant_no, tau, eta, alpha, beta):
    """
    Creates tours for all ants in the colony. Each ant constructs a complete tour
    probabilistically based on pheromone and desirability.
    """
    colony = Colony(ant_no)
    node_no = graph.n

    for i in range(ant_no):
        # Select a random initial node (0-based index)
        initial_node = np.random.randint(0, node_no)
        colony.ants[i].tour = [initial_node]
        current_node = initial_node
        visited_nodes = {initial_node} # Use a set for efficient lookup

        # Construct the primary tour visiting all nodes exactly once
        while len(visited_nodes) < node_no:
            current_node = colony.ants[i].tour[-1]

            # Calculate probabilities for next step
            # P_allNodes = (tau[current_node, :] ** alpha) * (eta[current_node, :] ** beta)
            # Ensure correct element-wise multiplication for numpy arrays
            pheromone_influence = tau[current_node, :] ** alpha
            desirability_influence = eta[current_node, :] ** beta
            P_allNodes = pheromone_influence * desirability_influence

            # Assign almost zero probability to visited nodes and infinite edges
            for j in range(node_no):
                if j in visited_nodes or np.isinf(graph.edges[current_node, j]):
                    P_allNodes[j] = 1e-10 # Small non-zero value to avoid division by zero if sum is 0

            # Normalize probabilities
            sum_P_allNodes = np.sum(P_allNodes)
            if sum_P_allNodes == 0: # Handle cases where no valid next move exists (shouldn't happen in a connected graph)
                unvisited = [n for n in range(node_no) if n not in visited_nodes and not np.isinf(graph.edges[current_node, n])]
                if unvisited:
                    next_node = np.random.choice(unvisited)
                else:
                    break
            else:
                P = P_allNodes / sum_P_allNodes
                next_node = roulette_wheel(P)

            colony.ants[i].tour.append(next_node)
            visited_nodes.add(next_node)

        # Complete the tour by returning to the initial node
        colony.ants[i].tour.append(initial_node) # Explicitly close the loop

    return colony


Update Pheromone Function The update_pheromone function modifies the pheromone matrix based on the tours completed by all ants in the current iteration. Ants that found shorter tours deposit more pheromone, reinforcing those paths. This mechanism drives the optimization process by strengthening promising routes.

In [10]:
def update_pheromone(tau, colony):
    """
    Updates the pheromone matrix based on the tours completed by the ants.
    Pheromone is deposited inversely proportional to the tour's fitness.
    """
    # Initialize delta_tau for this iteration
    delta_tau = np.zeros_like(tau)
    ant_no = len(colony.ants)

    for i in range(ant_no): # For each ant
        tour = colony.ants[i].tour
        fitness = colony.ants[i].fitness

        if fitness == 0: # Avoid division by zero for perfect tours (unlikely in TSP)
            pheromone_deposit = np.inf
        else:
            pheromone_deposit = 1.0 / fitness

        for j in range(len(tour) - 1): # For each edge in the tour (excluding the last return to start)
            node1 = tour[j]
            node2 = tour[j+1]

            # Deposit pheromone on both directions of the edge
            delta_tau[node1, node2] += pheromone_deposit
            delta_tau[node2, node1] += pheromone_deposit

    # Add the deposited pheromone to the existing pheromone matrix
    tau += delta_tau
    return tau

Get Graph Edges Trace (for Plotly) This helper function prepares the data for drawing all the graph edges using Plotly. It extracts the coordinates of connected nodes and formats them into a scatter trace suitable for line plotting.

In [11]:
def get_graph_edges_trace(graph, line_color='black', line_width=1, showlegend=False, name='Edges'):
    """Helper to create Plotly trace for all graph edges."""
    edge_x = []
    edge_y = []
    for i in range(graph.n):
        for j in range(i + 1, graph.n):
            if np.isfinite(graph.edges[i, j]):
                edge_x.extend([graph.nodes[i].x, graph.nodes[j].x, None])
                edge_y.extend([graph.nodes[i].y, graph.nodes[j].y, None])
    return go.Scatter(x=edge_x, y=edge_y, mode='lines', line=dict(color=line_color, width=line_width),
                      hoverinfo='none', showlegend=showlegend, name=name)

Get Graph Nodes Trace (for Plotly) This helper function prepares the data for drawing all the graph nodes using Plotly. It extracts the coordinates of each node and formats them into a scatter trace suitable for marker plotting, including node labels.

In [12]:
def get_graph_nodes_trace(graph, marker_color='cyan', marker_line_color='blue', marker_size=10, showlegend=False, name='Nodes'):
    """Helper to create Plotly trace for all graph nodes."""
    node_x = [node.x for node in graph.nodes]
    node_y = [node.y for node in graph.nodes]
    node_names = [f'Node {i+1}' for i in range(graph.n)]
    return go.Scatter(x=node_x, y=node_y, mode='markers+text', text=node_names, textposition="top center",
                      marker=dict(symbol='circle', size=marker_size, color=marker_color,
                                  line=dict(color=marker_line_color, width=1)),
                      hoverinfo='text', showlegend=showlegend, name=name)


Draw Graph Function The draw_graph function renders the basic structure of the graph, including all nodes and existing edges. It also has the capability to highlight a "deleted" edge, which will be used in the second part of the demonstration.

In [13]:
def draw_graph(fig, subplot_row, subplot_col, graph, edge_eliminated=False, eliminated_nodes=None):
    """
    Draws the graph (nodes and edges).
    """
    fig.add_trace(get_graph_edges_trace(graph, name='Graph Edges'), row=subplot_row, col=subplot_col)
    fig.add_trace(get_graph_nodes_trace(graph, name='Graph Nodes'), row=subplot_row, col=subplot_col)

    # Highlight eliminated edge if applicable
    if edge_eliminated and eliminated_nodes is not None:
        n1_idx, n2_idx = eliminated_nodes[0] - 1, eliminated_nodes[1] - 1 # Convert to 0-based
        fig.add_trace(go.Scatter(x=[graph.nodes[n1_idx].x, graph.nodes[n2_idx].x],
                                 y=[graph.nodes[n1_idx].y, graph.nodes[n2_idx].y],
                                 mode='lines', line=dict(color='lightgray', width=2, dash='dash'),
                                 name='Eliminated Edge', showlegend=False),
                      row=subplot_row, col=subplot_col)
    fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False, row=subplot_row, col=subplot_col)
    fig.update_yaxes(showgrid=False, zeroline=False, showticklabels=False, row=subplot_row, col=subplot_col)


Draw Best Tour Function The draw_best_tour function visualizes the optimal tour found by the "queen" ant (the best ant in the colony). It overlays the best tour in red on top of the faint gray background of all graph edges, making the optimal path clearly visible.

In [14]:
def draw_best_tour(fig, subplot_row, subplot_col, queen_tour, graph, edge_eliminated=False, eliminated_nodes=None):
    """
    Draws the best tour found by the queen ant.
    """
    # Draw all graph edges first (faintly)
    fig.add_trace(get_graph_edges_trace(graph, line_color='gray', line_width=0.5, name='All Edges'), row=subplot_row, col=subplot_col)

    # Draw the best tour
    tour_x = []
    tour_y = []
    for i in range(len(queen_tour) - 1):
        node1_idx = queen_tour[i]
        node2_idx = queen_tour[i+1]
        tour_x.extend([graph.nodes[node1_idx].x, graph.nodes[node2_idx].x, None])
        tour_y.extend([graph.nodes[node1_idx].y, graph.nodes[node2_idx].y, None])

    fig.add_trace(go.Scatter(x=tour_x, y=tour_y, mode='lines', line=dict(color='red', width=3),
                             name='Best Tour (Queen)'),
                  row=subplot_row, col=subplot_col)

    fig.add_trace(get_graph_nodes_trace(graph, name='Tour Nodes'), row=subplot_row, col=subplot_col)

    # Highlight eliminated edge if applicable
    if edge_eliminated and eliminated_nodes is not None:
        n1_idx, n2_idx = eliminated_nodes[0] - 1, eliminated_nodes[1] - 1 # Convert to 0-based
        fig.add_trace(go.Scatter(x=[graph.nodes[n1_idx].x, graph.nodes[n2_idx].x],
                                 y=[graph.nodes[n1_idx].y, graph.nodes[n2_idx].y],
                                 mode='lines', line=dict(color='lightgray', width=2, dash='dash'),
                                 name='Eliminated Edge', showlegend=False),
                      row=subplot_row, col=subplot_col)

    fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False, row=subplot_row, col=subplot_col)
    fig.update_yaxes(showgrid=False, zeroline=False, showticklabels=False, row=subplot_row, col=subplot_col)

Draw Pheromone Function The draw_pheromone function visualizes the pheromone concentration on the graph edges. The intensity of the blue color and the thickness of the lines are proportional to the pheromone levels, allowing us to observe how pheromone accumulates on promising paths over iterations.

In [15]:
def draw_pheromone(fig, subplot_row, subplot_col, tau, graph, edge_eliminated=False, eliminated_nodes=None):
    """
    Visualizes the pheromone concentration on the graph edges.
    Pheromone levels determine line width and color intensity.
    """
    max_tau = np.max(tau[np.isfinite(tau)]) if np.any(np.isfinite(tau)) else 1.0 # Handle case with no finite tau
    min_tau = np.min(tau[np.isfinite(tau)]) if np.any(np.isfinite(tau)) else 0.0

    # Draw all graph edges first (faintly)
    fig.add_trace(get_graph_edges_trace(graph, line_color='gray', line_width=0.5, name='All Edges'), row=subplot_row, col=subplot_col)

    # Draw pheromone trails
    for i in range(graph.n):
        for j in range(i + 1, graph.n):
            if np.isfinite(graph.edges[i, j]) and np.isfinite(tau[i, j]):
                # Normalize pheromone for visualization
                if max_tau == min_tau: # Avoid division by zero if all pheromone values are the same
                    tau_normalized = 0.5
                else:
                    tau_normalized = (tau[i, j] - min_tau) / (max_tau - min_tau)

                # Color: from light blue to dark blue (or black for very low)
                # Alpha (opacity) also based on normalized tau
                color_intensity = tau_normalized
                line_color = f'rgba(0, 0, {int(255 * color_intensity)}, {0.2 + 0.8 * tau_normalized})' # Adjust alpha for visibility
                line_width = 1 + 9 * tau_normalized # Line width from 1 to 10

                fig.add_trace(go.Scatter(x=[graph.nodes[i].x, graph.nodes[j].x],
                                         y=[graph.nodes[i].y, graph.nodes[j].y],
                                         mode='lines', line=dict(color=line_color, width=line_width),
                                         hoverinfo='text', text=f'Pheromone: {tau[i,j]:.2f}', showlegend=False),
                              row=subplot_row, col=subplot_col)

    fig.add_trace(get_graph_nodes_trace(graph, name='Pheromone Nodes'), row=subplot_row, col=subplot_col)

    # Highlight eliminated edge if applicable
    if edge_eliminated and eliminated_nodes is not None:
        n1_idx, n2_idx = eliminated_nodes[0] - 1, eliminated_nodes[1] - 1 # Convert to 0-based
        fig.add_trace(go.Scatter(x=[graph.nodes[n1_idx].x, graph.nodes[n2_idx].x],
                                 y=[graph.nodes[n1_idx].y, graph.nodes[n2_idx].y],
                                 mode='lines', line=dict(color='lightgray', width=2, dash='dash'),
                                 name='Eliminated Edge', showlegend=False),
                      row=subplot_row, col=subplot_col)

    fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False, row=subplot_row, col=subplot_col)
    fig.update_yaxes(showgrid=False, zeroline=False, showticklabels=False, row=subplot_row, col=subplot_col)


ACO Loop Function This is the core of the Ant Colony Optimization algorithm. The aco_loop function orchestrates the iterative process of tour construction by ants, evaluation of their tours, updating the pheromone matrix, and applying pheromone evaporation. It also tracks and displays the best tour found over iterations.

In [16]:
def aco_loop(graph, max_iter, ant_no, tau, eta, alpha, beta, rho, edge_eliminated=False, eliminated_nodes=None):
    """
    Main loop of the Ant Colony Optimization algorithm.
    Iteratively creates ant tours, updates pheromones, and performs evaporation.
    Returns the final pheromone matrix, best tour, AND the Plotly figure object.
    """
    best_fitness = np.inf
    best_tour = []

    # Create a single figure for dynamic updates
    fig = make_subplots(rows=1, cols=3, subplot_titles=("All Nodes and Edges", "Best Tour (The Queen)", "All Pheromones"))

    # Initial drawing for the first subplot
    draw_graph(fig, 1, 1, graph, edge_eliminated, eliminated_nodes)
    fig.update_layout(title_text="ACO Algorithm Progress", height=500, showlegend=False)

    for t in range(1, max_iter + 1):
        # Create Ants and their tours
        colony = create_colony(graph, ant_no, tau, eta, alpha, beta)

        # Calculate the fitness values of all ants
        for ant in colony.ants:
            ant.fitness = fitness_function(ant.tour, graph)

        # Find the best ant (queen) in the current iteration
        all_ants_fitness = [ant.fitness for ant in colony.ants]
        min_val = np.min(all_ants_fitness)
        min_index = np.argmin(all_ants_fitness)

        if min_val < best_fitness:
            best_fitness = min_val
            best_tour = colony.ants[min_index].tour

        colony.queen.tour = best_tour
        colony.queen.fitness = best_fitness

        # Update pheromone matrix based on current iteration's tours
        tau = update_pheromone(tau , colony)

        # Evaporation: Pheromone trails gradually evaporate
        tau = (1 - rho) * tau

        # Display results and visualize progress
        print(f"Iteration #{t}, Shortest length = {colony.queen.fitness:.2f}")

        # Update plots every 25 iterations or at the last iteration
        if t % 25 == 0 or t == max_iter:
            # Clear existing traces in the subplots before redrawing
            # We keep the first subplot (All Nodes and Edges) as it doesn't change
            fig.data = [trace for trace in fig.data if trace.xaxis == 'x1']

            draw_best_tour(fig, 1, 2, colony.queen.tour, graph, edge_eliminated, eliminated_nodes)
            draw_pheromone(fig, 1, 3, tau, graph, edge_eliminated, eliminated_nodes)

            fig.update_layout(title_text=f"ACO Algorithm Progress (Iteration {t})", height=500, showlegend=False)
            # No fig.show() here. The figure will be shown once at the end of the main execution block.

    return tau, best_tour, fig # Return the figure object

Main Execution - Part 1: Initial Optimization This section sets up the problem parameters and executes the first phase of the ACO algorithm. It performs 500 iterations to find an initial optimal tour for the given graph. The final state of the graph, the best tour, and pheromone distribution will be displayed.

In [17]:
# --- Problem Preparation ---
edge_eliminated = False
deleted_nodes = [0, 0] # Placeholder for 1-based node indices

x_coords = [7, 5, 7, 9, 4, 6, 8, 10, 3, 5, 7, 9, 11, 4, 6, 8, 10, 5, 7, 9, 7]
y_coords = [1, 3, 3, 3, 5, 5, 5, 5, 7, 7, 7, 7, 7, 9, 9, 9, 9, 11, 11, 11, 13]
graph = create_graph(x_coords, y_coords)

# --- ACO Algorithm - First Part ---
print("--- Part 1: Initial ACO Optimization ---")

# Initial parameters of ACO
max_iter = 500
ant_no = 50
# Initial pheromone concentration
finite_edges = graph.edges[np.isfinite(graph.edges)]
if len(finite_edges) > 0:
    tau0 = 10 * 1 / (graph.n * np.mean(finite_edges))
else:
    tau0 = 1.0 # Default if no finite edges (shouldn't happen for a connected graph)

tau = np.zeros((graph.n, graph.n)) # Pheromone matrix
finite_edges_indices = np.isfinite(graph.edges)
tau[finite_edges_indices] = tau0

eta = np.zeros_like(graph.edges) # Desirability of each edge (inverse of distance)
eta[finite_edges_indices] = 1.0 / graph.edges[finite_edges_indices]

rho = 0.8      # Evaporation rate
alpha = 0.5    # Pheromone exponential parameters
beta = 0.5     # Desirability exponential parameter

# Main loop of ACO
# aco_loop now returns the figure object
tau, queen_tour, first_fig = aco_loop(graph, max_iter, ant_no, tau, eta, alpha, beta, rho, edge_eliminated, deleted_nodes)

# Display the final figure of the first part after the loop completes
print("\n--- First optimization phase completed ---")
print("Displaying the final graph after 500 iterations.")
first_fig.show() # Display the figure here


--- Part 1: Initial ACO Optimization ---
Iteration #1, Shortest length = inf
Iteration #2, Shortest length = inf
Iteration #3, Shortest length = 114.00
Iteration #4, Shortest length = 114.00
Iteration #5, Shortest length = 114.00
Iteration #6, Shortest length = 114.00
Iteration #7, Shortest length = 104.00
Iteration #8, Shortest length = 104.00
Iteration #9, Shortest length = 100.00
Iteration #10, Shortest length = 100.00
Iteration #11, Shortest length = 100.00
Iteration #12, Shortest length = 100.00
Iteration #13, Shortest length = 100.00
Iteration #14, Shortest length = 100.00
Iteration #15, Shortest length = 100.00
Iteration #16, Shortest length = 100.00
Iteration #17, Shortest length = 100.00
Iteration #18, Shortest length = 100.00
Iteration #19, Shortest length = 100.00
Iteration #20, Shortest length = 100.00
Iteration #21, Shortest length = 100.00
Iteration #22, Shortest length = 100.00
Iteration #23, Shortest length = 100.00
Iteration #24, Shortest length = 100.00
Iteration #25,

Main Execution - Part 2: Adapting to Graph Changes This section demonstrates a key feature of ACO: its ability to efficiently adapt to changes in the problem domain by reusing the existing pheromone matrix. After an edge is "deleted" (simulated by setting its weight to infinity), the algorithm re-optimizes the tour. You will be prompted to select an edge to remove from the graph.

In [18]:
print("\n" * 3) # Add some empty lines to push the prompt down for visibility
print("--- Part 2: Adapting to Graph Changes ---")
print("Demonstrating ACO's ability to re-optimize after an edge deletion.")
print("Please choose an edge to eliminate from the list below, referring to the graph above.")
print("Scroll down if necessary to see the list and the input box.")

# --- Interactive Edge Selection ---
available_edges = []
edge_counter = 1
for i in range(graph.n):
    for j in range(i + 1, graph.n):
        if np.isfinite(graph.edges[i, j]):
            available_edges.append(((i, j), edge_counter))
            print(f"{edge_counter}. Node {i+1} - Node {j+1}")
            edge_counter += 1

selected_edge_nodes = None
while True:
    try:
        # Add explicit message and instruction for the user
        print("\n" * 2) # Add some empty lines before the prompt
        print(">>> ATTENTION: The input box might appear at the bottom of the output. <<<")
        print(">>> If you don't see it, please scroll down or click on the output area. <<<")
        choice_input = input("Enter the number of the edge to eliminate: ")
        choice_idx = int(choice_input)

        if not (1 <= choice_idx <= len(available_edges)):
            print(f"Invalid choice. Please enter a number between 1 and {len(available_edges)}. Try again.")
            continue

        chosen_edge_tuple, _ = available_edges[choice_idx - 1]
        node_to_delete_1, node_to_delete_2 = chosen_edge_tuple

        # Store 1-based indices for visualization function consistency
        deleted_nodes = [node_to_delete_1 + 1, node_to_delete_2 + 1]

        # Delete the identified edge by setting its weight to infinity
        graph.edges[node_to_delete_1, node_to_delete_2] = np.inf
        graph.edges[node_to_delete_2, node_to_delete_1] = np.inf
        edge_eliminated = True
        print(f"Edge between Node {node_to_delete_1 + 1} and Node {node_to_delete_2 + 1} has been 'eliminated'.")
        break # Exit loop if input is valid
    except ValueError:
        print("Invalid input. Please enter an integer number. Try again.")
    except Exception as e:
        print(f"An error occurred: {e}. Please try again.")


# New parameters for ACO (reusing the last pheromone matrix)
max_iter = 250 # Fewer iterations for re-optimization
# Increase initial pheromone concentration slightly to encourage re-exploration
tau[np.isfinite(graph.edges)] = tau[np.isfinite(graph.edges)] + tau0 # Re-add tau0 to existing finite edges

# Recalculate desirability as edge weights might have changed
eta = np.zeros_like(graph.edges)
finite_edges_indices_after_deletion = np.isfinite(graph.edges)
eta[finite_edges_indices_after_deletion] = 1.0 / graph.edges[finite_edges_indices_after_deletion]

# New loops of ACO
tau, queen_tour, second_fig = aco_loop(graph, max_iter, ant_no, tau, eta, alpha, beta, rho, edge_eliminated, deleted_nodes)

print("\n--- Optimization Completed ---")
print(f"Final best tour length: {fitness_function(queen_tour, graph):.2f}")
print(f"Final best tour (0-based indices): {queen_tour}")
print(f"Final best tour (1-based indices): {[node + 1 for node in queen_tour]}")
second_fig.show() # Display the final figure of the second phase here





--- Part 2: Adapting to Graph Changes ---
Demonstrating ACO's ability to re-optimize after an edge deletion.
Please choose an edge to eliminate from the list below, referring to the graph above.
Scroll down if necessary to see the list and the input box.
1. Node 1 - Node 2
2. Node 1 - Node 3
3. Node 1 - Node 4
4. Node 2 - Node 3
5. Node 2 - Node 5
6. Node 2 - Node 6
7. Node 3 - Node 4
8. Node 3 - Node 6
9. Node 3 - Node 7
10. Node 4 - Node 7
11. Node 4 - Node 8
12. Node 5 - Node 6
13. Node 5 - Node 9
14. Node 5 - Node 10
15. Node 6 - Node 7
16. Node 6 - Node 10
17. Node 6 - Node 11
18. Node 7 - Node 8
19. Node 7 - Node 11
20. Node 7 - Node 12
21. Node 8 - Node 12
22. Node 8 - Node 13
23. Node 9 - Node 10
24. Node 9 - Node 14
25. Node 10 - Node 11
26. Node 10 - Node 14
27. Node 10 - Node 15
28. Node 11 - Node 12
29. Node 11 - Node 15
30. Node 11 - Node 16
31. Node 12 - Node 13
32. Node 12 - Node 16
33. Node 12 - Node 17
34. Node 13 - Node 17
35. Node 14 - Node 15
36. Node 14 - Node 